In [2]:

import rasterio
import numpy as np

In [3]:


with rasterio.open(r"D:\Resume Projs\EcoRouteAI\Dataset\Processed Data\slope.tif") as src:
    vh = src.read(1)
    print("Scales:", src.scales)
    print("Offsets:", src.offsets)
    print(src.profile)
    print(src.tags())
    print(src.descriptions)
    print(src.units)
'''
print("Min:", np.nanmin(vh))
print("Max:", np.nanmax(vh))
print("Mean:", np.nanmean(vh))
print("Dtype:", vh.dtype)
for p in [0,1,5,25,50,75,95,99,100]:
    print(p, np.percentile(vh, p))'''

Scales: (1.0,)
Offsets: (0.0,)
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': None, 'width': 2461, 'height': 1316, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 45N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",87],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32645"]]'), 'transform': Affine(28.79349362284501, 0.0, 435431.20666596334,
       0.0, -28.79349362284501, 2648758.370621523), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
{'AREA_OR_POINT': 'Area'}
(None,)
(Non

'\nprint("Min:", np.nanmin(vh))\nprint("Max:", np.nanmax(vh))\nprint("Mean:", np.nanmean(vh))\nprint("Dtype:", vh.dtype)\nfor p in [0,1,5,25,50,75,95,99,100]:\n    print(p, np.percentile(vh, p))'

In [3]:
with rasterio.open(r"D:\Resume Projs\EcoRouteAI\Dataset\Processed Data\Labels.tif") as src:
    print("Bands:", src.count)
    print("Dtype:", src.dtypes)
    print("NoData:", src.nodata)
    data = src.read()
    print("Unique values:", np.unique(data))
    print("Shape:", src.shape)

Bands: 1
Dtype: ('uint8',)
NoData: 0.0
Unique values: [10 20 30 40 50 60 80 90]
Shape: (1316, 2461)


In [ ]:


FUSED_STACK_PATH = r"Dataset\Fused Sentinel Datafused_clipped.tif"
DEM_PATH = r"Dataset\DEM\DEM.tiff"

with rasterio.open(FUSED_STACK_PATH) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("\nAffine Transform:")
    print(src.transform)



def inspect_raster(path, name):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    with rasterio.open(path) as src:
        print(f"  CRS        : {src.crs}")
        print(f"  Shape      : {src.height} rows x {src.width} cols")
        print(f"  Resolution : {src.res}")
        print(f"  Bounds (native CRS):")
        print(f"    Left   = {src.bounds.left:.2f}")
        print(f"    Right  = {src.bounds.right:.2f}")
        print(f"    Bottom = {src.bounds.bottom:.2f}")
        print(f"    Top    = {src.bounds.top:.2f}")
        
        # Convert corners to WGS84 lat/lon
        transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
        
        corners = {
            "Bottom-Left" : (src.bounds.left,  src.bounds.bottom),
            "Top-Right"   : (src.bounds.right, src.bounds.top),
        }
        print(f"  Corners in WGS84 (Lon, Lat):")
        for label, (x, y) in corners.items():
            lon, lat = transformer.transform(x, y)
            print(f"    {label}: Lat={lat:.4f}, Lon={lon:.4f}")
        
        print(f"  Band count : {src.count}")
        print(f"  Dtype      : {src.dtypes[0]}")
        print(f"  NoData     : {src.nodata}")

inspect_raster(FUSED_STACK_PATH, "FUSED STACK")
inspect_raster(DEM_PATH, "DEM (original)")

In [1]:
import time
import pickle

start = time.time()

with open(r"D:\Resume Projs\EcoRouteAI\Dataset\costGraph.gpickle", "rb") as src:
    g = pickle.load(src)
    MIN_EDGE_COST = min(
    d['weight'] for u, v, d in g.edges(data=True) if d['weight'] != float('inf'))
    print(MIN_EDGE_COST)

print(time.time() - start)

1.0
24.151110649108887


In [10]:
import random
import time
import networkx as nx

rows = 1316
cols = 2461
minCost = 2

def manhattenHeur(a,b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


test_routes = [
    
      ((296, 1491), (806, 405)),
      ((769, 1832), (1312, 790)),
      ((143, 2407) , (122, 367)),
      ((331, 2050), (444, 130)),
      ((505, 1409) , (870, 1333)),
      ((1300, 1746), (296, 120)), 
      ((478, 975), (1103, 641)),
      ((1298, 2445),(982, 942)),
      ((886, 723),(1292, 1810)),
      ((119, 1135),(655, 1669)),
    
]
i=1
for src, dst in test_routes:
    start = time.time()

    path = nx.astar_path(
        g,
        source=src,
        target=dst,
        heuristic= manhattenHeur,
        weight="weight"
    )

    elapsed = time.time() - start

    print(
        f"Test {i}: "
        f"{src} → {dst} | "
        f"{len(path)} nodes | "
        f"{elapsed:.3f} sec"
    )
    i +=1 

Test 1: (296, 1491) → (806, 405) | 1597 nodes | 19.548 sec
Test 2: (769, 1832) → (1312, 790) | 1886 nodes | 29.981 sec
Test 3: (143, 2407) → (122, 367) | 2184 nodes | 16.732 sec
Test 4: (331, 2050) → (444, 130) | 2034 nodes | 23.545 sec
Test 5: (505, 1409) → (870, 1333) | 496 nodes | 1.068 sec
Test 6: (1300, 1746) → (296, 120) | 2777 nodes | 5.382 sec
Test 7: (478, 975) → (1103, 641) | 988 nodes | 5.722 sec
Test 8: (1298, 2445) → (982, 942) | 2334 nodes | 9.760 sec
Test 9: (886, 723) → (1292, 1810) | 1644 nodes | 13.289 sec
Test 10: (119, 1135) → (655, 1669) | 1595 nodes | 2.866 sec


In [4]:
import random
import time
import networkx as nx
test_routes = [
    (
         (331, 2154),(22, 172)
    )
    for _ in range(2)
]
minCost=2
def manhattenHeur(a,b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def manhxmtcHeur(a,b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])*minCost
for i, (src, dst) in enumerate(test_routes, start=1):
    start = time.time()

    path = nx.astar_path(
        G,
        source=src,
        target=dst,
        heuristic= manhxmtcHeur,
        weight="weight"
    )

    elapsed = time.time() - start

    print(
        f"Test {i}: "
        f"{src} → {dst} | "
        f"{len(path)} nodes | "
        f"{elapsed:.3f} sec"
    )

KeyboardInterrupt: 